# 06 - Bronze Clickstream

            Uses Auto Loader with `addNewColumns`, rescued data, an append-only
            Delta target, schema-version logging, quarantine, and idempotent
            available-now micro-batches.

In [ ]:
%run ./05_bronze_common

In [ ]:
import uuid
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("landing_root", "/Volumes/abc_retail/dev/landing", "Landing root")
dbutils.widgets.text(
    "checkpoint_root",
    "/Volumes/abc_retail/dev/landing/_checkpoints",
    "Checkpoint root",
)
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
landing_root = dbutils.widgets.get("landing_root").strip().rstrip("/")
checkpoint_root = dbutils.widgets.get("checkpoint_root").strip().rstrip("/")
job_run_id = dbutils.widgets.get("job_run_id").strip()

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`bronze`")

source_name = "clickstream_events"
pipeline_name = "bronze_clickstream"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
source_root = f"{landing_root}/{source_name}"
bronze_table = f"{catalog}.bronze.bronze_clickstream"
quarantine_table = f"{catalog}.quarantine.clickstream_errors"
schema_location = f"{checkpoint_root}/schemas/{pipeline_name}"
checkpoint_location = f"{checkpoint_root}/streams/{pipeline_name}"

upsert_audit(
    catalog,
    pipeline_run_id,
    pipeline_name,
    "available_now",
    start_ts,
    "RUNNING",
    [source_root],
    job_run_id,
)

In [ ]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`bronze`.`bronze_clickstream` (
        raw_payload STRING,
        _rescued_data STRING,
        schema_version STRING,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        load_date DATE NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    PARTITIONED BY (load_date)
    TBLPROPERTIES (
        'delta.appendOnly' = 'true',
        'comment' = 'Append-only raw clickstream records with Auto Loader schema evolution'
    )
    """
)

In [ ]:
def process_clickstream_batch(batch_df, micro_batch_id: int) -> None:
    unprocessed, files_df = filter_unprocessed_stream_batch(
        batch_df, catalog, source_name
    )
    if files_df.limit(1).count() == 0:
        return

    _, transaction_version = batch_token(files_df)
    metadata_columns = {"source_file", "file_size", "modification_time"}
    payload_columns = [
        F.col(column_name).alias(column_name)
        for column_name in unprocessed.columns
        if column_name not in metadata_columns
    ]
    schema_json = unprocessed.drop(
        "source_file", "file_size", "modification_time"
    ).schema.json()
    schema_version = log_schema_version(
        catalog,
        source_name,
        schema_json,
        files_df,
        pipeline_run_id,
    )

    corrupt_payload_expr = (
        F.col("_corrupt_record")
        if "_corrupt_record" in unprocessed.columns
        else F.get_json_object("_rescued_data", "$._corrupt_record")
    )

    prepared = (
        add_lineage(
            unprocessed.withColumn(
                "raw_payload", F.to_json(F.struct(*payload_columns))
            ),
            pipeline_run_id,
        )
        .withColumn("schema_version", F.lit(schema_version))
        .withColumn(
            "corrupt_payload",
            corrupt_payload_expr,
        )
        .withColumn(
            "event_time_check",
            F.expr("try_to_timestamp(event_time)"),
        )
        .withColumn(
            "error_code",
            F.when(F.col("corrupt_payload").isNotNull(), F.lit("parse_error"))
            .when(F.col("event_id").isNull(), F.lit("missing_key"))
            .when(F.col("event_time_check").isNull(), F.lit("invalid_timestamp")),
        )
        .withColumn(
            "error_reason",
            F.when(F.col("error_code") == "parse_error", F.lit("Malformed JSON payload"))
            .when(F.col("error_code") == "missing_key", F.lit("event_id is null"))
            .when(
                F.col("error_code") == "invalid_timestamp",
                F.lit("event_time cannot be parsed"),
            ),
        )
    )

    valid = prepared.filter(F.col("error_code").isNull()).drop(
        "file_size",
        "modification_time",
        "corrupt_payload",
        "event_time_check",
        "error_code",
        "error_reason",
    )

    errors = (
        prepared.filter(F.col("error_code").isNotNull())
        .select(
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("source_file"),
                    F.coalesce(F.col("event_id"), F.lit("")),
                    F.col("raw_payload"),
                    F.col("error_code"),
                ),
                256,
            ).alias("error_id"),
            F.lit("bronze").alias("error_stage"),
            "error_code",
            "error_reason",
            F.col("event_id").alias("record_key"),
            F.coalesce("corrupt_payload", "raw_payload").alias("raw_payload"),
            F.col("_rescued_data").alias("rescued_data"),
            "source_file",
            "batch_id",
            "load_date",
            "ingest_ts",
            F.current_date().alias("quarantine_date"),
            "pipeline_run_id",
        )
    )

    valid_count = append_delta(
        valid,
        bronze_table,
        f"{source_name}_bronze",
        transaction_version,
        merge_schema=True,
    )
    error_count = append_delta(
        errors,
        quarantine_table,
        f"{source_name}_quarantine",
        transaction_version,
    )
    error_counts = errors.groupBy("error_code").count().collect()
    write_dq_metrics(
        catalog,
        source_name,
        pipeline_run_id,
        transaction_version,
        valid_count,
        error_counts,
    )
    mark_files_success(catalog, source_name, files_df, pipeline_run_id)
    print(
        f"micro_batch={micro_batch_id}, valid={valid_count}, "
        f"quarantined={error_count}"
    )

In [ ]:
try:
    stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(source_root)
        .select(
            "*",
            F.col("_metadata.file_path").alias("source_file"),
            F.col("_metadata.file_size").alias("file_size"),
            F.col("_metadata.file_modification_time").alias("modification_time"),
        )
    )

    query = (
        stream.writeStream.foreachBatch(process_clickstream_batch)
        .option("checkpointLocation", checkpoint_location)
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()

    processed_paths = [
        row.source_file
        for row in spark.table(f"{catalog}.control.processed_files")
        .filter(F.col("pipeline_run_id") == pipeline_run_id)
        .select("source_file")
        .distinct()
        .collect()
    ]

    metrics = (
        spark.table(f"{catalog}.control.data_quality_results")
        .filter(F.col("pipeline_run_id") == pipeline_run_id)
        .agg(
            F.sum(
                F.when(F.col("quality_status") == "valid", F.col("record_count")).otherwise(0)
            ).alias("rows_written"),
            F.sum(
                F.when(F.col("quality_status") == "error", F.col("record_count")).otherwise(0)
            ).alias("rows_quarantined"),
        )
        .first()
    )
    rows_written = int(metrics.rows_written or 0)
    rows_quarantined = int(metrics.rows_quarantined or 0)
    upsert_audit(
        catalog,
        pipeline_run_id,
        pipeline_name,
        "available_now",
        start_ts,
        "SUCCESS",
        processed_paths or [source_root],
        job_run_id,
        end_ts=datetime.now(timezone.utc),
        rows_read=rows_written + rows_quarantined,
        rows_written=rows_written,
        rows_quarantined=rows_quarantined,
    )
except Exception as exc:
    upsert_audit(
        catalog,
        pipeline_run_id,
        pipeline_name,
        "available_now",
        start_ts,
        "FAILED",
        [source_root],
        job_run_id,
        end_ts=datetime.now(timezone.utc),
        error_message=str(exc)[:4000],
    )
    raise

In [ ]:
spark.table(bronze_table).groupBy("load_date").count().orderBy("load_date").display()